<a href="https://colab.research.google.com/github/Vinsmoke-R/6-chains_langchain/blob/main/Transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from nltk.tokenize import word_tokenize
import nltk

In [ ]:
# Positional Encoding (embedding)
def tokenize(text):
  text = text.lower()
  text = text.replace('?','')
  text = text.replace("'","")
  return text.split()

In [ ]:
vocab = {'UNK':0}

In [ ]:
def build_vocab(text):
  tokenized_text = tokenize(text)

  for token in tokenized_text:
    if token not in vocab:
      vocab[token] = len(vocab)

In [ ]:
build_vocab("hellow My name is Raj Yadav")
vocab

{'UNK': 0, 'hellow': 1, 'my': 2, 'name': 3, 'is': 4, 'raj': 5, 'yadav': 6}

In [ ]:
def text_to_indices(sentence,vocab):
  tokens = tokenize(sentence)        # split into words , so easy to iterate
  numerical_sentence = []

  for token in tokens:
    if token in vocab:
      numerical_sentence.append(vocab[token])
    else:
      numerical_sentence.append(vocab['UNK'])

  return numerical_sentence

In [ ]:
# padding
def padded_text(text):

  padded_text = torch.tensor(text + [0] * (512 - len(text)) , dtype=torch.long).unsqueeze(0)
  return padded_text

In [ ]:
a = text_to_indices("bankai katen kyōkotsu karamatsu shinjū",vocab)
a = padded_text(a)
a

tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0

In [ ]:
def embedded(vocab_size, embedding_dim):

  embedding_matrix = torch.randn(vocab_size, embedding_dim, requires_grad=True)
  return embedding_matrix


def get_embeddings(token_ids, embedding_matrix):

  return embedding_matrix[token_ids]

In [ ]:
embedding_matrix = embedded(len(vocab), 512)
embedded_tokens = get_embeddings(a, embedding_matrix)

In [ ]:
embedded_tokens.shape

torch.Size([1, 512, 512])

In [ ]:
#Positional Encoding
import math
def pos(sequence_length, d_model):

  PE = [[0 for _ in range(d_model)] for _ in range(sequence_length)]

  for j in range(0,sequence_length):
    for i in range(0,d_model):
      if i%2==0:
        PE[j][i] = math.sin(j/math.pow(10000,(2*i)/d_model))
      else:
        PE[j][i] = math.cos(j/math.pow(10000,(2*i)/d_model))

  PE = torch.tensor(PE, dtype=torch.float32)
  return PE

In [ ]:
PE = pos(512,512)

In [ ]:
len(PE[0])

512

In [ ]:
def add_positional_encoding(embedded_tokens, PE):
    # (1, seq_len, d_model) + (seq_len, d_model)
    return embedded_tokens + PE.unsqueeze(0)   #we can do loop but this is faster and easy to do

In [ ]:
first_input = add_positional_encoding(embedded_tokens,PE)

In [ ]:
first_input

tensor([[[ 0.1715,  2.0229,  2.0565,  ...,  1.4307, -0.1396,  0.6171],
         [ 1.0130,  1.5925,  2.8585,  ...,  1.4307, -0.1396,  0.6171],
         [ 1.0808,  0.6720,  3.0147,  ...,  1.4307, -0.1396,  0.6171],
         ...,
         [ 0.2335,  1.6252,  2.7154,  ...,  1.4307, -0.1396,  0.6171],
         [ 1.0449,  0.7100,  1.8469,  ...,  1.4307, -0.1396,  0.6171],
         [ 1.0533,  0.0640,  1.1471,  ...,  1.4307, -0.1396,  0.6171]]],
       grad_fn=<AddBackward0>)

In [ ]:
first_input.shape

torch.Size([1, 512, 512])

In [ ]:
PE.dtype

torch.float32

In [ ]:
embedded_tokens.dtype

torch.float32

In [ ]:
# self attention
class self_attention(nn.Module):
  def __init__(self,d_model,d_k,d_v):
    super().__init__()

    self.W_Q = nn.Linear(d_model,d_k)
    self.W_K = nn.Linear(d_model,d_k)
    self.W_V = nn.Linear(d_model,d_v)
    self.scale = math.sqrt(d_k)

  def forward(self,Q,K,V,mask = None):   # X: input embedding {shape = (batch_size, seq_len, d_model)}

    Q = self.W_Q(Q)
    K = self.W_K(K)
    V = self.W_V(V)

    attention_scores = torch.matmul(Q, K.transpose(-2, -1)) / self.scale  # (1,a,b) -> (1,b,a) {(1,512,512) -> (1,512,512)}

    if mask is not None:
            attention_scores = attention_scores.masked_fill(~mask, float('-inf'))

    attention_weights = torch.softmax(attention_scores, dim=-1)  # (batch, seq_len, seq_len)

    # Weighted sum of values
    attention_output = torch.matmul(attention_weights, V)  # (batch, seq_len, d_v)

    return attention_output

In [ ]:
# multi head attention
class multi_head_attention(nn.Module):
  def __init__(self):
    super().__init__()
    self.x1 = self_attention(512,64,64)
    self.x2 = self_attention(512,64,64)
    self.x3 = self_attention(512,64,64)
    self.x4 = self_attention(512,64,64)
    self.x5 = self_attention(512,64,64)
    self.x6 = self_attention(512,64,64)
    self.x7 = self_attention(512,64,64)
    self.x8 = self_attention(512,64,64)
    self.W_O = nn.Linear(64 * 8, 512)

  def forward(self,Q,K,V,mask=None):
    o1 = self.x1(Q,K,V,mask=mask)
    o2 = self.x2(Q,K,V,mask=mask)
    o3 = self.x3(Q,K,V,mask=mask)
    o4 = self.x4(Q,K,V,mask=mask)
    o5 = self.x5(Q,K,V,mask=mask)
    o6 = self.x6(Q,K,V,mask=mask)
    o7 = self.x7(Q,K,V,mask=mask)
    o8 = self.x8(Q,K,V,mask=mask)
    y = torch.cat([o1,o2,o3,o4,o5,o6,o7,o8], dim=2) #(batch_size, seq_length, features)
    output = self.W_O(y)
    return output

In [ ]:
multi_head_output = multi_head_attention()
second_input = multi_head_output(first_input)

In [ ]:
second_input

tensor([[[ 0.0898, -0.1681,  0.7609,  ...,  0.1023, -0.3235,  0.3569],
         [ 0.0893, -0.1677,  0.7611,  ...,  0.1017, -0.3232,  0.3569],
         [ 0.0889, -0.1673,  0.7614,  ...,  0.1006, -0.3222,  0.3566],
         ...,
         [ 0.0897, -0.1688,  0.7636,  ...,  0.1024, -0.3215,  0.3562],
         [ 0.0899, -0.1690,  0.7633,  ...,  0.1028, -0.3218,  0.3561],
         [ 0.0902, -0.1692,  0.7630,  ...,  0.1028, -0.3221,  0.3560]]],
       grad_fn=<ViewBackward0>)

In [ ]:
second_input.shape

torch.Size([1, 512, 512])

In [ ]:
third_input = first_input + second_input

In [ ]:
third_input.shape

torch.Size([1, 512, 512])

In [ ]:
layer_norm = nn.LayerNorm(512)

In [ ]:
fourth_input = layer_norm(third_input)

In [ ]:
# feed forward layer
class feed_forward(nn.Module):
  def __init__(self,num_features):
    super().__init__()
    self.model = nn.Sequential(
        nn.Linear(num_features,2048),
        nn.ReLU(),
        nn.Linear(2048,512)
    )
  def forward(self,x):
    return self.model(x)


In [ ]:
ff = feed_forward(512)

In [ ]:
fifth_input = ff(fourth_input)

In [ ]:
fifth_input

tensor([[[ 0.1690,  0.2046,  0.1138,  ..., -0.2601, -0.0346,  0.1135],
         [ 0.1736,  0.2356,  0.0938,  ..., -0.2435, -0.0556,  0.0810],
         [ 0.2105,  0.2523,  0.0775,  ..., -0.2338, -0.0782,  0.0462],
         ...,
         [ 0.0134,  0.1724,  0.3344,  ..., -0.3694, -0.0129,  0.1159],
         [-0.0017,  0.1839,  0.3614,  ..., -0.3466, -0.0127,  0.1534],
         [-0.0166,  0.1939,  0.3726,  ..., -0.3393, -0.0078,  0.1516]]],
       grad_fn=<ViewBackward0>)

In [ ]:
fifth_input.shape

torch.Size([1, 512, 512])

In [ ]:
sixth_input = fourth_input + fifth_input

In [ ]:
sixth_input

tensor([[[-0.1022,  1.3013,  2.0371,  ...,  0.5604, -0.9277,  0.4541],
         [ 0.6092,  0.9493,  2.7004,  ...,  0.5630, -0.9699,  0.4060],
         [ 0.7033,  0.1711,  2.8198,  ...,  0.5711, -0.9935,  0.3698],
         ...,
         [-0.0816,  1.0156,  2.8521,  ...,  0.5373, -0.7572,  0.5592],
         [ 0.5800,  0.2709,  2.1726,  ...,  0.5674, -0.7611,  0.6005],
         [ 0.5800, -0.2583,  1.6129,  ...,  0.5847, -0.7593,  0.6050]]],
       grad_fn=<AddBackward0>)

In [ ]:
sixth_input.shape

torch.Size([1, 512, 512])

In [ ]:
seventh_input = layer_norm(sixth_input)

In [ ]:
seventh_input

tensor([[[-0.0953,  1.2727,  1.9898,  ...,  0.5506, -0.9000,  0.4469],
         [ 0.6004,  0.9324,  2.6420,  ...,  0.5553, -0.9412,  0.4020],
         [ 0.6932,  0.1731,  2.7612,  ...,  0.5639, -0.9648,  0.3673],
         ...,
         [-0.0765,  0.9955,  2.7901,  ...,  0.5282, -0.7368,  0.5496],
         [ 0.5695,  0.2673,  2.1264,  ...,  0.5572, -0.7415,  0.5896],
         [ 0.5689, -0.2515,  1.5799,  ...,  0.5735, -0.7419,  0.5934]]],
       grad_fn=<NativeLayerNormBackward0>)

In [ ]:
seventh_input.shape

torch.Size([1, 512, 512])

In [ ]:
# Encoder block
class EncoderBlock(nn.Module):
  def __init__(self):
    super().__init__()
    self.mha = multi_head_attention()
    self.norm1 = nn.LayerNorm(512)
    self.ff = feed_forward(512)
    self.norm2 = nn.LayerNorm(512)

  def forward(self,x):
    mha_output = self.mha(x,x,x,mask=None)
    x = self.norm1(mha_output + x)

    ff_output = self.ff(x)
    x = self.norm2(ff_output + x)

    return x

In [ ]:
# Encoder layers
class Encoder(nn.Module):
  def __init__(self):
    super().__init__()
    self.x1 = EncoderBlock()
    self.x2 = EncoderBlock()
    self.x3 = EncoderBlock()
    self.x4 = EncoderBlock()
    self.x5 = EncoderBlock()
    self.x6 = EncoderBlock()

  def forward(self,x):
    o1 = self.x1(x)
    o2 = self.x2(o1)
    o3 = self.x3(o2)
    o4 = self.x4(o3)
    o5 = self.x5(o4)
    o6 = self.x6(o5)

    return o6

In [ ]:
# Decoder Block
class DecoderBlock(nn.Module):
  def __init__(self):
    super().__init__()
    self.mha1 = multi_head_attention()
    self.norm1 = nn.LayerNorm(512)

    self.mha2 = multi_head_attention()
    self.norm2 = nn.LayerNorm(512)

    self.ff = feed_forward(512)
    self.norm3 = nn.LayerNorm(512)

  def forward(self,x,encoder_output):

    seq_len = x.size(1)
    mask = torch.tril(torch.ones(seq_len, seq_len)).bool()
    mha1_output = self.mha1(x,x,x,mask=mask)
    x = self.norm1(mha1_output + x)

    mha2_output = self.mha2(x,encoder_output,encoder_output,mask=None)
    x = self.norm2(mha2_output+x)

    ff_output = self.ff(x)
    x = self.norm3(ff_output + x)

    return x

In [ ]:
#decoder layer
class Decoder(nn.Module):
  def __init__(self):
    super().__init__()
    self.x1 = DecoderBlock()
    self.x2 = DecoderBlock()
    self.x3 = DecoderBlock()
    self.x4 = DecoderBlock()
    self.x5 = DecoderBlock()
    self.x6 = DecoderBlock()

  def forward(self,x,encoder_output):
    o1 = self.x1(x,encoder_output)
    o2 = self.x2(o1,encoder_output)
    o3 = self.x3(o2,encoder_output)
    o4 = self.x4(o3,encoder_output)
    o5 = self.x5(o4,encoder_output)
    o6 = self.x6(o5,encoder_output)

    return o6

In [ ]:
# Output layers
class Transformer(nn.Module):
  def __init__(self, vocab_size, d_model=512, seq_len=512):
    super().__init__()
    self.embed_tokens = nn.Embedding(vocab_size, d_model)
    self.pos_encoding = pos(seq_len, d_model)
    self.encoder = Encoder()
    self.decoder = Decoder()
    self.linear = nn.Linear(d_model, vocab_size)

  def forward(self,x,y):

    encoder_input = self.embed_tokens(x)
    encoder_input = encoder_input + self.pos_encoding.unsqueeze(0)
    encoder_output = self.encoder(encoder_input)

    decoder_input = self.embed_tokens(y)
    decoder_input = decoder_input + self.pos_encoding.unsqueeze(0)
    decoder_output = self.decoder(decoder_input,encoder_output)

    output = self.linear(decoder_output)

    return output

In [ ]:
model = Transformer(vocab_size)

In [ ]:
PAD_IDX = 0
d_model = 512
step_num = 4000
warmup_steps = 4000
params = model.parameters()

In [ ]:
# loss and optimizer
loss = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

optimizer = optim.Adam(model.parameters(), betas=(0.9, 0.98), eps=1e-9)

def get_lr(step, d_model=512, warmup_steps=4000):
    return (d_model ** -0.5) * min(step ** -0.5, step * (warmup_steps ** -1.5))

# we manually update lr in each step here
for step in range(1, 10000):
    lr = get_lr(step, d_model, warmup_steps)         # compute lr for this step
    for param_group in optimizer.param_groups:       # update optimizer lr
        param_group['lr'] = lr

In [ ]:
# train the model


In [ ]:
# Evalute & test the transformer